In [1]:
import asyncio
import xmlschema
import networkx as nx
import polars as pl
from itertools import zip_longest

In [2]:
from ggblab import GeoGebra
ggb = await GeoGebra().init(use_vscode=False)

In [3]:
# ggb.file.load('eg10_slider3.ggb')
ggb.file.load('2025_06_08.ggb')

In [4]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [5]:
from ggblab_extra import ConstructionIO
from ggblab_extra import ConstructionTreeParser

In [6]:

df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)
df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
u32,str,str,str,str,str,u32,bool,bool,bool
1,"""O_{2}""","""point""",null,"""O_{2} = (-2.1, -0.1)""",null,9,true,true,false
2,"""A""","""point""",null,"""A = (10, 0)""",null,2,true,true,false
3,"""B""","""point""","""Point(Line(O_{2}, A))""","""B = (-9.6, -0.2)""",null,9,true,false,false
4,"""c""","""circle""","""Circle(O_{2}, B)""","""c: (x + 2.1)² + (y + 0.1)² = 5…",null,9,true,false,false
5,"""n""","""line""","""Line(O_{2}, A)""","""n: y = -0.1""",null,4,false,false,false
6,"""O'""","""point""","""Midpoint(O_{2}, A)""","""O' = (3.9, 0)""",null,4,false,true,false
7,"""p""","""circle""","""Circle(O', A)""","""p: (x - 3.9)² + (y + 0)² = 36.…",null,4,false,false,false
8,"""j""","""line""","""Tangent(A, c)""","""j: -5.8x - 7.6y = -58.2""","""$a+c$""",2,true,false,false
9,"""l""","""line""","""Tangent(A, c)""","""l: 5.9x - 7.5y = 59.4""",null,2,true,false,false


In [8]:
p = ConstructionTreeParser(df)
g1 = p.parse()
nx.write_network_text(g1)

╟── O_{2}
╎   ├─╼ B ╾ A
╎   │   └─╼ c ╾ O_{2}
╎   │       ├─╼ j ╾ A
╎   │       │   ├─╼ q ╾ l
╎   │       │   ├─╼ r ╾ l
╎   │       │   │   └─╼ O_{1}
╎   │       │   │       ├─╼ t ╾ l
╎   │       │   │       │   ├─╼ F_{1} ╾ l
╎   │       │   │       │   │   ├─╼ h_1 ╾ O_{1}
╎   │       │   │       │   │   │   ├─╼ p_2 ╾ O_{2}, c
╎   │       │   │       │   │   │   │   ├─╼ q_2 ╾ O_{1}
╎   │       │   │       │   │   │   │   │   └─╼ N ╾ p_2
╎   │       │   │       │   │   │   │   │       └─╼ d_2 ╾ O_{2}
╎   │       │   │       │   │   │   │   │           └─╼ T ╾ c
╎   │       │   │       │   │   │   │   ├─╼ r_2 ╾ O_{1}
╎   │       │   │       │   │   │   │   │   └─╼ M ╾ p_2
╎   │       │   │       │   │   │   │   │       └─╼ b_2 ╾ O_{2}
╎   │       │   │       │   │   │   │   │           └─╼ F_{2} ╾ c
╎   │       │   │       │   │   │   │   │               ├─╼ γ ╾ A, O_{2}
╎   │       │   │       │   │   │   │   │               ├─╼ t2 ╾ A, O_{2}
╎   │       │   │       │   │   │   │   │   

In [9]:
g2 = p.parse_subgraph_legacy()
nx.write_network_text(g2)

╟── A
╎   └─╼ B ╾ O_{2}
╎       └─╼ c
╎           ├─╼ j
╎           │   └─╼ r ╾ l
╎           │       └─╼ O_{1}
╎           │           ├─╼ t
╎           │           │   └─╼ F_{1}
╎           │           │       └─╼ h_1
╎           │           │           ├─╼ k_1
╎           │           │           ├─╼ k_2
╎           │           │           │   ├─╼ n_2
╎           │           │           │   └─╼ m_2
╎           │           │           ├─╼ p_2
╎           │           │           │   ├─╼ q_2
╎           │           │           │   │   └─╼ N
╎           │           │           │   └─╼ r_2
╎           │           │           │       └─╼ M
╎           │           │           │           └─╼ b_2
╎           │           │           │               └─╼ F_{2}
╎           │           │           └─╼ j_1
╎           │           ├─╼ O_3
╎           │           │   └─╼ p_1
╎           │           │       ├─╼ C
╎           │           │       ├─╼ D
╎           │           │       └─╼ E
╎           

In [11]:
l1 = 'k'
l2 = 'n_1'

In [12]:
await ggb.listen(l1, True)
await ggb.listen(l2, True)

{}

In [13]:
# await ggb.listen('a', False)
# await ggb.listen('b', False)

In [21]:
ggb.comm.shared_objects

{'k': 'k = 3', 'n_1': 'n_1 = 7'}

In [15]:
# import ipywidgets as widgets
# label1 = widgets.Label(value=ggb.comm.shared_objects['n'])
# label2 = widgets.Label(value=ggb.comm.shared_objects['m'])
# display(label1, label2)

In [28]:
async def on_shared_update1(changes):
    # await asyncio.sleep(0)
    # label1.value = changes['a']
    n = int(changes[l1].split()[2])
    # await asyncio.sleep(0)
    await ggb.function("setLayerVisible", list(zip_longest(range(9), [True]*n, fillvalue=False)))
    r = await ggb.function('getXML', [l2])
    o = ggb.file.ggb_schema.decode(r)
    o['value'][0]['@val'] = '0'
    x = xmlschema.etree_tostring(ggb.file.ggb_schema.encode(o, 'element'))
    r = await ggb.function('evalXML' , [x])

In [29]:
ggb.comm.remove_shared_listener(on_shared_update1)
ggb.comm.add_shared_listener(on_shared_update1)

True

In [30]:
async def on_shared_update2(changes):
    # await asyncio.sleep(0)
    # label2.value = changes['b']
    m = int(changes[l2].split()[2])
    n = int(ggb.comm.shared_objects[l1].split()[2])
    l = df.filter(pl.col("Layer") == n)["Name"].to_list()
    # m = int(ggb.comm.shared_objects['m'].split()[2])
    # list(zip_longest(l, [True]*m, fillvalue=False))
    await ggb.function("setVisible", list(zip_longest(l, [True]*m, fillvalue=False)))

In [31]:
ggb.comm.remove_shared_listener(on_shared_update2)
ggb.comm.add_shared_listener(on_shared_update2)

True

In [27]:
ggb.comm.clear_shared_listeners()

2